<a href="https://colab.research.google.com/github/Elliotn46/27-13001506-DE-JS-Assesment/blob/main/seismic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install obspy ipython-sql sqlalchemy folium pandas matplotlib

import os
import obspy
import sqlite3
from obspy.clients.fdsn import Client
import pandas as pd
import matplotlib.pyplot as plt
import folium


os.makedirs("database", exist_ok=True)
os.makedirs("data", exist_ok=True)
%load_ext sql
%sql sqlite:///database/seismic.db

In [2]:
%%sql
--Stations Table
CREATE TABLE IF NOT EXISTS stations (
    network TEXT NOT NULL,
    station TEXT NOT NULL,
    latitude REAL NOT NULL,
    longitude REAL NOT NULL,
    elevation REAL,
    PRIMARY KEY (network, station)
);

--Channels Table
CREATE TABLE IF NOT EXISTS channels (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    network TEXT NOT NULL,
    station TEXT NOT NULL,
    location TEXT NOT NULL,
    channel TEXT NOT NULL,
    sampling_rate REAL NOT NULL,
    FOREIGN KEY (network, station) REFERENCES stations (network, station) ON DELETE CASCADE,
    UNIQUE (network, station, location, channel)
);

--Waveform Headers / Metadata Table
CREATE TABLE IF NOT EXISTS waveforms (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    channel_id INTEGER NOT NULL,
    starttime TEXT NOT NULL,
    endtime TEXT NOT NULL,
    npts INTEGER NOT NULL,
    file_source TEXT NOT NULL,
    FOREIGN KEY (channel_id) REFERENCES channels (id) ON DELETE CASCADE
);

 * sqlite:///database/seismic.db
Done.
Done.
Done.


[]

In [3]:
# Initialize SQLite database
db_path = "database/seismic.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Execute schema creation
cursor.execute('''CREATE TABLE IF NOT EXISTS stations (network TEXT, station TEXT, latitude REAL, longitude REAL, elevation REAL, PRIMARY KEY (network, station));''')
cursor.execute('''CREATE TABLE IF NOT EXISTS channels (id INTEGER PRIMARY KEY AUTOINCREMENT, network TEXT, station TEXT, location TEXT, channel TEXT, sampling_rate REAL);''')
conn.commit()

# Load mseed files
mseed_files = ["/SEP01.mseed", "/SEP02.mseed", "/SEP03.mseed"]
iris_client = Client("EARTHSCOPE")

# Track discovered stations to query IRIS once
discovered_stations = set()

for file_path in mseed_files:
    if os.path.exists(file_path):
        st = obspy.read(file_path, format="MSEED")

        # Insert station metadata
        for tr in st:
            net, sta, loc, cha = tr.stats.network, tr.stats.station, tr.stats.location, tr.stats.channel
            discovered_stations.add((net, sta))

            # Insert channel metadata
            cursor.execute('''INSERT OR IGNORE INTO channels (network, station, location, channel, sampling_rate) VALUES (?, ?, ?, ?, ?)''',
                           (net, sta, loc, cha, tr.stats.sampling_rate))
        conn.commit()

print(f"Processed miniSEED files. Discovered stations from data: {discovered_stations}")

Processed miniSEED files. Discovered stations from data: {('CC', 'SEP')}


In [4]:
import obspy
import sqlite3
import os

db_path = "database/seismic.db"
mseed_files = ["/SEP01.mseed", "/SEP02.mseed", "/SEP03.mseed"]

with sqlite3.connect(db_path) as conn:
    cursor = conn.cursor()

    # Enforce foreign keys in SQLite
    cursor.execute("PRAGMA foreign_keys = ON;")

    for file_path in mseed_files:
        if not os.path.exists(file_path):
            print(f"⚠️ Skipping {file_path} (not found)")
            continue

        st = obspy.read(file_path)

        for tr in st:
            stats = tr.stats
            net = stats.network
            sta = stats.station
            loc = stats.location
            cha = stats.channel
            sampling_rate = stats.sampling_rate
            start_time = str(stats.starttime)
            end_time = str(stats.endtime)
            npts = stats.npts

            # 1. Insert into stations (uses composite PK: network, station)
            cursor.execute("""
                INSERT OR IGNORE INTO stations (network, station, latitude, longitude, elevation)
                VALUES (?, ?, ?, ?, ?)
            """, (net, sta, 0.0, 0.0, 0.0))

            # 2. Insert into channels (references network and station)
            cursor.execute("""
                INSERT OR IGNORE INTO channels (network, station, location, channel, sampling_rate)
                VALUES (?, ?, ?, ?, ?)
            """, (net, sta, loc, cha, sampling_rate))

            # Fetch the auto-incremented channel id
            cursor.execute("""
                SELECT id FROM channels
                WHERE network = ? AND station = ? AND location = ? AND channel = ?
            """, (net, sta, loc, cha))

            channel_row = cursor.fetchone()
            if not channel_row:
                continue
            channel_id = channel_row[0]

            # 3. Insert into waveforms (matches your table definition)
            cursor.execute("""
                INSERT INTO waveforms (channel_id, starttime, endtime, npts, file_source)
                VALUES (?, ?, ?, ?, ?)
            """, (channel_id, start_time, end_time, npts, os.path.basename(file_path)))

    conn.commit()

print("✅ Database successfully populated with your schema!")

✅ Database successfully populated with your schema!
